<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/NLP_Assessment_3_Filtering_and_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateBERT Filtering & Scoring
**Pipeline:** combined.csv → climate-relevance filter → sentiment scoring → outputs

Pulls `combined.csv` directly from GitHub — no Drive mounting, no reprocessing.

**Models used:**
- `climatebert/distilroberta-base-climate-detector` — filters out non-climate text
- `climatebert/distilroberta-base-climate-sentiment` — scores remaining text positive/negative/neutral

**Outputs pushed to `data/processed/` on GitHub:**
- `filtered.csv` — climate-relevant rows only
- `individual_scores.csv` — per-row sentiment scores
- `company_summary.csv` — average sentiment per company (greenwashing signal)

**Before running:**
1. Click the "KEY" icon in the left sidebar
2. Add a secret called `GITHUB_TOKEN` with your personal access token
   - Get one at: github.com/settings/tokens → Generate new token (classic) → tick `repo`
3. Toggle **Notebook access** on

In [ ]:
# --- SETUP ---
!pip install transformers torch sentence-transformers -q

import os
import pandas as pd
import torch
from google.colab import userdata
from transformers import AutoTokenizer

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

OUTPUT_DIR = '/content/outputs'
REPO       = 'Swas26/NLP-ESG'
REPO_DIR   = '/content/repo'
GIT_EMAIL  = 'swas.2625@gmial.com'
GIT_NAME   = 'Swas26'

os.makedirs(OUTPUT_DIR, exist_ok=True)

!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR} -q 2>/dev/null || echo "Repo already cloned"

combined_path = f'{REPO_DIR}/data/processed/combined.csv'
if not os.path.exists(combined_path):
    raise FileNotFoundError('combined.csv not found — run Data_Preprocessing notebook first')

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
print('Setup complete')

In [ ]:
# --- LOAD DATA ---

df = pd.read_csv(combined_path)
print(f'Loaded {len(df)} rows from GitHub')
print(df['source_type'].value_counts())
print('\nPer company:')
print(df.groupby(['company','source_type']).size().unstack(fill_value=0))

In [ ]:
# --- PART 1: CLIMATE RELEVANCE FILTERING ---
from transformers import pipeline

detector = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-detector',
    device=0 if torch.cuda.is_available() else -1
)

tokenizer = AutoTokenizer.from_pretrained('climatebert/distilroberta-base-climate-detector')

def is_climate_related(text):
    try:
        result = detector(str(text), truncation=True, max_length=512)[0]
        return result['label'] == 'yes'
    except Exception as e:
        return False


print('Filtering with ClimateBERT detector...')
df['is_relevant'] = df['text'].apply(is_climate_related)

before = len(df)
filtered_df = df[df['is_relevant'] == True].drop(columns=['is_relevant']).reset_index(drop=True)
after = len(filtered_df)

print(f'Kept {after} / {before} rows ({round(after/before*100)}% climate-relevant)')
print(filtered_df['source_type'].value_counts())
print('\nPer company:')
print(filtered_df.groupby(['company','source_type']).size().unstack(fill_value=0))

filtered_df.to_csv(f'{OUTPUT_DIR}/filtered.csv', index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-detector
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Filtering with ClimateBERT detector...


In [ ]:
# --- PART 2: SENTIMENT SCORING ---

torch.cuda.empty_cache()

sentiment = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-sentiment',
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

def get_sentiment_score(text):
    try:
        results = sentiment(str(text), truncation=True, max_length=512)[0]
        score_dict = {item['label'].lower(): item['score'] for item in results}
        p_opp  = score_dict.get('opportunity', 0)
        p_risk = score_dict.get('risk', 0)
        return round(p_opp - p_risk, 4)
    except Exception as e:
        print(f'Error: {e}')
        return None

print('\nRunning ClimateBERT sentiment scoring...')
filtered_df['sentiment_score'] = filtered_df['text'].apply(get_sentiment_score)

scored = filtered_df['sentiment_score'].notna().sum()
print(f'Scored {scored} / {len(filtered_df)} rows')

filtered_df.to_csv(f'{OUTPUT_DIR}/individual_scores.csv', index=False)

In [ ]:
results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

print(results.groupby(['company', 'source_type'])['sentiment_score'].mean().round(4).unstack())

In [ ]:
# --- PART 2B: RoBERTa COMPARISON ---

torch.cuda.empty_cache()

roberta_pipe = pipeline(
    'text-classification',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

def get_roberta_score(text):
    try:
        results = roberta_pipe(str(text), truncation=True, max_length=512)[0]
        score_dict = {item['label'].lower(): item['score'] for item in results}
        return round(score_dict.get('positive', 0) - score_dict.get('negative', 0), 4)
    except Exception as e:
        print(f'Error: {e}')
        return None

results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

print('Running RoBERTa baseline scoring...')
results['roberta_score'] = results['text'].apply(get_roberta_score)
print(f"Scored {results['roberta_score'].notna().sum()} / {len(results)} rows")

results.to_csv(f'{OUTPUT_DIR}/individual_scores.csv', index=False)

print('\n=== ClimateBERT vs RoBERTa ===')
print(results.groupby('company')[['sentiment_score', 'roberta_score']].mean().round(4))

In [ ]:

VAGUE_MARKERS = [
    'committed to', 'aim to', 'working towards', 'we believe',
    'intend to', 'strive to', 'aspire', 'our journey', 'we recognise',
    'dedicated to', 'our goal is', 'we hope', 'in line with',
    'we support', 'we acknowledge', 'sustainability is core',
    'we are focused on', 'we continue to', 'we remain committed',
    'we are proud', 'we look forward', 'we are working',
]

SPECIFIC_MARKERS = [
    'reduced by', 'increased by', 'achieved', 'certified',
    'verified by', 'third party', 'independently audited',
    'scope 1', 'scope 2', 'scope 3', 'tonnes', 'megawatts',
    'invested $', 'per cent', 'compared to', 'baseline',
    'target of', 'by 2030', 'by 2035', 'by 2040', 'by 2050',
    'fy2023', 'fy2024', 'fy2025', 'in 2023', 'in 2024',
    '%', 'kwh', 'gwh', 'co2e', 'mtco2'
]

def specificity_score(text):
    text_lower = str(text).lower()
    vague_hits    = sum(1 for v in VAGUE_MARKERS    if v in text_lower)
    specific_hits = sum(1 for s in SPECIFIC_MARKERS if s in text_lower)
    total = vague_hits + specific_hits
    if total == 0:
        return 0.0
    return round((specific_hits - vague_hits) / total, 4)

results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

results['specificity_score'] = results['text'].apply(specificity_score)

results.to_csv(f'{OUTPUT_DIR}/individual_scores.csv', index=False)


In [ ]:
# --- PART 4: DIVERGENCE SCORE ---
results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

report_stats = (
    results[results['source_type'] == 'esg_report']
    .groupby('company')
    .agg(
        report_sentiment=('sentiment_score', 'mean'),
        report_chunks=('sentiment_score', 'count'),
    )
    .round(4)
)

news_stats = (
    results[results['source_type'] == 'news']
    .groupby('company')
    .agg(
        news_sentiment=('sentiment_score', 'mean'),
        news_chunks=('sentiment_score', 'count'),
    )
    .round(4)
)

divergence = report_stats.join(news_stats)

divergence['divergence_score'] = (
    divergence['report_sentiment'] - divergence['news_sentiment']
).round(4)

MIN_REPORT_CHUNKS = 20
MIN_NEWS_CHUNKS   = 5
divergence['data_quality'] = 'ok'
divergence.loc[divergence['report_chunks'] < MIN_REPORT_CHUNKS, 'data_quality'] = 'insufficient_report_data'
divergence.loc[divergence['news_chunks']   < MIN_NEWS_CHUNKS,   'data_quality'] = 'insufficient_news_data'
divergence.loc[divergence['news_sentiment'].isna(), 'data_quality'] = 'insufficient_news_data'

divergence = divergence.sort_values('divergence_score', ascending=False)

print('=== DIVERGENCE SCORE (report_sentiment − news_sentiment) ===')
print(divergence[['report_sentiment', 'news_sentiment', 'divergence_score',
                   'report_chunks', 'news_chunks', 'data_quality']].to_string())

divergence.to_csv(f'{OUTPUT_DIR}/divergence_scores.csv')
print('\nDivergence scores saved')

In [ ]:
# --- PART 4B: RoBERTa DIVERGENCE INDEX (for comparison only) ---

results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

roberta_report = (
    results[results['source_type'] == 'esg_report']
    .groupby('company')['roberta_score'].mean().round(4)
    .rename('roberta_report')
)
roberta_news = (
    results[results['source_type'] == 'news']
    .groupby('company')['roberta_score'].mean().round(4)
    .rename('roberta_news')
)

rob_div = pd.concat([roberta_report, roberta_news], axis=1)
rob_div['roberta_greenwashing_index'] = (
    rob_div['roberta_report'] - rob_div['roberta_news']
).round(4)

divergence = pd.read_csv(f'{OUTPUT_DIR}/divergence_scores.csv', index_col='company')

comparison = divergence[['divergence_score']].join(rob_div[['roberta_greenwashing_index']])
comparison.columns = ['ClimateBERT_DS', 'RoBERTa_DS']

print('=== Side-by-side Divergence Score ===')
print(comparison.sort_values('ClimateBERT_DS', ascending=False).to_string())


In [ ]:
# --- PART 5: GREENWASHING INDICATOR ---

import numpy as np

results    = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')
divergence = pd.read_csv(f'{OUTPUT_DIR}/divergence_scores.csv', index_col='company')

DIVERGENCE_THRESHOLD  = 0.5
SPECIFICITY_THRESHOLD = 0.4

# def sigmoid(x):
#     return 1 / (1 + np.exp(-x))

avg_spec   = results.groupby('company')['specificity_score'].mean().rename('avg_specificity')
divergence = divergence.join(avg_spec)
# divergence['gw_probability'] = divergence['divergence_score'].apply(sigmoid)
reliable   = divergence[divergence['data_quality'] == 'ok'].copy()

print(f'\n=== CLASSIFICATION  ===')
print(f'thresholds: divergence > {DIVERGENCE_THRESHOLD}  |  specificity < {SPECIFICITY_THRESHOLD}\n')

for company, row in reliable.iterrows():
    ds   = row['divergence_score']
    spec = row['avg_specificity']
    high_divergence = ds   > DIVERGENCE_THRESHOLD
    low_specificity = spec < SPECIFICITY_THRESHOLD

    if high_divergence and low_specificity:
        flag = 'POTENTIAL GREENWASHING  (high divergence + low specificity)'
    elif high_divergence:
        flag = 'ELEVATED RISK           (high divergence)'
    elif low_specificity:
        flag = 'WATCH                   (low specificity)'
    else:
        flag = 'LOW RISK'

    print(f"  {company:<25}  DS={ds:+.3f}  spec={spec:.3f}  gw_prob={row['divergence_score']:.3f}  ->  {flag}")

In [ ]:
# --- PART 6: GREENWASHING SECTOR BINNING ---

!python -m spacy download en_core_web_sm -q
import spacy
from collections import defaultdict


flagged = reliable[
    (reliable['divergence_score'] > DIVERGENCE_THRESHOLD) |
    (reliable['avg_specificity']  < SPECIFICITY_THRESHOLD)
]
flagged_companies = flagged.index.tolist()



nlp = spacy.load('en_core_web_sm')

SECTOR_KEYWORDS = {
    'Emissions & Carbon': [
        'scope 1', 'scope 2', 'scope 3', 'greenhouse gas', 'ghg', 'net zero',
        'carbon offset', 'decarbonisation', 'emissions reduction', 'co2-e',
        'carbon footprint', 'flag', 'forest land agriculture', 'methane',
        'nitrous oxide', 'refrigerant', 'global warming potential', 'gwp',
        'science-based targets', 'sbti', 'paris agreement', 'carbon credit',
        'tco2e', 'emission intensity', 'carbon workshop', 'baseline emissions',
        'net zero 2050', 'emission inventory', 'carbon accounting'
    ],
    'Energy': [
        'renewable electricity', 'solar', 'wind farm', 'lgc', 'large-scale generation certificate',
        'energy efficiency', 'hvac', 'refrigeration', 'natural refrigerant',
        'onsite solar', 'power purchase agreement', 'electricity consumption',
        'energy use', 'mwh', 'kwh', 'megawatt', 'kilowatt', 'electrification',
        'electric vehicle', 'ev charging', 'battery storage', 'grid electricity',
        'renewable energy', 'origin zero', 'solar panel', 'clean energy',
        'flexible load', 'demand-based', 'energy transition', 'load shifting'
    ],
    'Water': [
        'water usage', 'water consumption', 'water efficiency', 'water management',
        'water meter', 'rainwater harvesting', 'water reuse', 'water quality',
        'dissolved oxygen', 'water availability', 'low-flow', 'water recycling',
        'water intensity', 'irrigation', 'water stress', 'freshwater',
        'macquarie harbour', 'tidal flow', 'seagrass', 'blue carbon',
        'coastal wetland', 'water footprint', 'water baseline'
    ],
    'Waste & Circular Economy': [
        'waste diversion', 'landfill', 'recycling', 'food waste', 'circular economy',
        'packaging', 'single-use plastic', 'soft plastics', 'recyclable',
        'compostable', 'reusable', 'recycled content', 'product stewardship',
        'container deposit', 'battery recycling', 'e-waste', 'organics',
        'food donation', 'secondbite', 'foodbank', 'waste hierarchy',
        'resource recovery', 'kerbside recyclable', 'arl', 'apco',
        'process engineered fuel', 'pef', 'plastic removal', 'packaging recyclability',
        'solid waste', 'waste reduction', 'waste stream', 'redcycle'
    ],
    'Biodiversity & Land': [
        'biodiversity', 'deforestation', 'nature positive', 'ecosystem',
        'habitat', 'forest', 'land use', 'no deforestation', 'palm oil',
        'rspo', 'soy', 'beef', 'cocoa', 'timber', 'pulp', 'paper',
        'accountability framework', 'afi', 'flag emissions', 'land conversion',
        'maugean skate', 'salmon farming', 'aquaculture', 'seagrass restoration',
        'great barrier reef', 'reef foundation', 'nature roadmap', 'tnfd',
        'leap framework', 'nature-related', 'species', 'endangered', 'marine ecosystem',
        'microbat', 'pesticide', 'on-farm biodiversity', 'tree planting', 'conservation plan'
    ],
    'Social & Labour': [
        'human rights', 'living wage', 'labour standards', 'modern slavery',
        'gender pay gap', 'pay parity', 'diversity', 'inclusion', 'indigenous',
        'aboriginal', 'torres strait islander', 'lgbtqi+', 'disability',
        'reconciliation', 'cultural awareness', 'wellbeing', 'mental health',
        'safety', 'trifr', 'injury', 'workplace safety', 'sexual harassment',
        'respect@work', 'engagement score', 'team member', 'leadership',
        'women in leadership', 'equity', 'fair wages', 'working conditions',
        'freedom of association', 'child labour', 'forced labour', 'grievance',
        'remediation', 'community support', 'food insecurity', 'healthy food'
    ],
    'Supply Chain': [
        'supplier engagement', 'ethical sourcing', 'responsible sourcing',
        'supply chain', 'audit', 'unannounced audit', 'supplier target',
        'traceability', 'certification', 'msc', 'asc', 'rspca', 'fairtrade',
        'rainforest alliance', 'globalg.a.p', 'bap', 'fsc', 'pefc',
        'bonsucro', 'rtrs', 'iscc', 'proaterra', 'eudr', 'cargill',
        'supplier portal', 'nurture fund', 'producer', 'farmer',
        'livestock', 'dairy', 'animal welfare', 'cage-free', 'sow stall free',
        'antibiotics', 'growth promotant', 'five freedoms', 'five domains',
        'backhaul', 'transport fleet', 'direct sourcing', 'supply chain emissions'
    ],
    'Governance & Policy': [
        'governance', 'board', 'materiality', 'gri', 'assurance', 'ernst young',
        'sustainability strategy', 'sustainability report', 'asrs', 'tcfd',
        'tnfd', 'ungc', 'un global compact', 'sdg', 'sustainable development goals',
        'stakeholder engagement', 'disclosure', 'transparency', 'accountability',
        'executive leadership', 'audit committee', 'risk management',
        'steering committee', 'working group', 'policy', 'compliance',
        'regulatory', 'mandatory reporting', 'climate-related financial disclosure',
        'science based targets initiative', 'sbti validation', 'greenwashing',
        'target setting', 'kpi', 'performance monitoring', 'annual report'
    ],
}

def assign_sector(text):
    text_lower = str(text).lower()
    hits = {sector: sum(1 for kw in kws if kw in text_lower)
            for sector, kws in SECTOR_KEYWORDS.items()}
    best = max(hits, key=hits.get)
    return best if hits[best] > 0 else 'Other'

def extract_ner_types(text):
    doc = nlp(str(text)[:10000])
    return [ent.label_ for ent in doc.ents]

flagged_companies = flagged.index.tolist()
print(f'Flagged companies: {flagged_companies}\n')

results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

esg_flagged = results[
    (results['company'].isin(flagged_companies)) &
    (results['source_type'] == 'esg_report')
].copy()

print(f'Total ESG chunks from flagged companies: {len(esg_flagged)}')

print('Assigning sectors...')
esg_flagged['sector'] = esg_flagged['text'].apply(assign_sector)

print('Running NER...')
esg_flagged['ner_types'] = esg_flagged['text'].apply(extract_ner_types)

esg_flagged.to_csv(f'{OUTPUT_DIR}/sector_analysis.csv', index=False)

print('\n=== CHUNKS PER SECTOR PER COMPANY ===\n')
pivot = esg_flagged.groupby(['company', 'sector']).size().unstack(fill_value=0)
print(pivot.to_string())

print('\n=== TOTAL CHUNKS PER SECTOR (all flagged companies) ===\n')
print(esg_flagged['sector'].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

try:
    import squarify
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'squarify', '-q'])
    import squarify

esg_flagged = pd.read_csv(f'{OUTPUT_DIR}/sector_analysis.csv')

pivot         = esg_flagged.groupby(['company', 'sector']).size().unstack(fill_value=0)
sector_totals = esg_flagged['sector'].value_counts().sort_values()
companies     = list(pivot.index)
sectors       = list(pivot.columns)

fig = plt.figure(figsize=(22, 12), facecolor='white')

ax1       = fig.add_subplot(2, 2, 1)
ax2_dummy = fig.add_subplot(2, 2, 3)

ax3 = fig.add_subplot(1, 2, 2)

for ax in [ax1, ax2_dummy, ax3]:
    ax.set_facecolor('white')

ax1.barh(range(len(sector_totals)), sector_totals.values,
         color='#4A7C8E', edgecolor='white', height=0.6)
ax1.set_yticks(range(len(sector_totals)))
ax1.set_yticklabels(sector_totals.index, fontsize=10)
ax1.set_xlabel('Flagged Chunks', fontsize=10)
ax1.set_title('Total Flagged Chunks\nby ESG Sector', fontsize=12,
              fontweight='bold', pad=10)

for i, v in enumerate(sector_totals.values):
    ax1.text(v + 2, i, str(v), va='center', fontsize=9, color='#333333')

ax1.set_xlim(0, sector_totals.max() * 1.15)
ax1.spines[['top', 'right', 'left']].set_visible(False)
ax1.grid(axis='x', color='#EBEBEB', linewidth=0.7)
ax1.set_axisbelow(True)

ax2_dummy.set_visible(False)
main_pos = ax2_dummy.get_position()

n_companies = len(companies)
pad         = 0.02
h_each      = (main_pos.height - pad * (n_companies - 1)) / n_companies

sub_axes = []
for i, company in enumerate(companies):
    bottom = main_pos.y0 + (n_companies - 1 - i) * (h_each + pad)
    ax = fig.add_axes([main_pos.x0, bottom, main_pos.width, h_each])
    ax.set_facecolor('white')
    sub_axes.append(ax)

sector_colors = {s: c for s, c in zip(sectors, plt.cm.Set2.colors)}

for i, (ax_sub, company) in enumerate(zip(sub_axes, companies)):
    row = pivot.loc[company].sort_values(ascending=True)
    row = row[row > 0]

    bar_colors = [sector_colors.get(s, '#AAAAAA') for s in row.index]
    ax_sub.barh(range(len(row)), row.values,
                color=bar_colors, edgecolor='white', height=0.6)
    ax_sub.set_yticks(range(len(row)))
    ax_sub.set_yticklabels(row.index, fontsize=7.5)
    ax_sub.spines[['top', 'right', 'left']].set_visible(False)
    ax_sub.grid(axis='x', color='#EBEBEB', linewidth=0.6)
    ax_sub.set_axisbelow(True)
    ax_sub.set_xlim(0, pivot.values.max() * 1.15)

    for j, v in enumerate(row.values):
        ax_sub.text(v + 0.5, j, str(v), va='center', fontsize=7)

    ax_sub.set_title(company, fontsize=9, fontweight='bold',
                     loc='left', pad=3, color='#333333')
    if i < n_companies - 1:
        ax_sub.set_xticklabels([])
    else:
        ax_sub.set_xlabel('Flagged Chunks', fontsize=9)

fig.text(main_pos.x0 + main_pos.width / 2,
         main_pos.y0 + main_pos.height + 0.02,
         'Sector Breakdown per Company',
         ha='center', va='bottom', fontsize=12, fontweight='bold',
         transform=fig.transFigure)

records = []
for company in companies:
    for sector in sectors:
        val = pivot.loc[company, sector]
        if val >= 5:
            records.append({
                'label':  f"{company}\n{sector}\n({val})",
                'size':   val,
                'sector': sector,
            })

df_tree       = pd.DataFrame(records).sort_values('size', ascending=False)
unique_sectors = df_tree['sector'].unique()
cmap_tree      = plt.cm.Set2
sec_color_map  = {s: cmap_tree(i / len(unique_sectors))
                  for i, s in enumerate(unique_sectors)}
tree_colors    = [sec_color_map[s] for s in df_tree['sector']]

squarify.plot(
    sizes=df_tree['size'].values,
    label=df_tree['label'].values,
    color=tree_colors,
    alpha=0.88,
    ax=ax3,
    text_kwargs={'fontsize': 10, 'color': '#1a1a1a'}
)
ax3.set_axis_off()
ax3.set_title('Greenwashing Hotspots by Company & Sector\n(tiles ≥ 5 chunks)',
              fontsize=12, fontweight='bold', pad=12)

patches = [mpatches.Patch(color=sec_color_map[s], label=s)
           for s in unique_sectors]
ax3.legend(handles=patches, fontsize=8.5, frameon=False,
           ncol=2, loc='lower center',
           bbox_to_anchor=(0.5, -0.06))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sector_analysis.png',
            dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved sector_analysis.png')

In [ ]:
# --- PUSH OUTPUTS TO GITHUB ---
import shutil

processed_dir = f'{REPO_DIR}/data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.csv'):
        shutil.copy(f'{OUTPUT_DIR}/{f}', f'{processed_dir}/{f}')
        print(f'Copied {f}')

!cd {REPO_DIR} && git config user.email "{GIT_EMAIL}"
!cd {REPO_DIR} && git config user.name "{GIT_NAME}"
!cd {REPO_DIR} && git add data/processed/
!cd {REPO_DIR} && git commit -m "update scoring outputs with divergence and cheap talk scores"
!cd {REPO_DIR} && git push

print('\nAll outputs pushed to GitHub → data/processed/')